# 🚀 Kaggle Remote GPU Server
### Exposes this Kaggle notebook as an SSH-accessible GPU server via Cloudflare Tunnel

**Instructions:**
1. Enable GPU accelerator in notebook settings (Settings → Accelerator → GPU T4 x2 or P100)
2. Add your SSH public key as a Kaggle Secret named `SSH_PUBLIC_KEY`
3. Run all cells — the tunnel URL will be printed at the end
4. Use the printed `ssh_config` block in your local `~/.ssh/config`

> ⚠️ Kaggle sessions last up to **12 hours** (GPU) or **9 hours** (TPU). Re-run when session expires.

## Verify GPU

In [ ]:
import subprocess, os, time, threading, re, json

# Verify GPU availability
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:')
    for line in result.stdout.strip().split('\n'):
        print(f'   {line}')
else:
    print('⚠️  No GPU found. Go to Settings → Accelerator to enable GPU.')

print(f'\n📁 Working directory: {os.getcwd()}')
print(f'👤 User: {subprocess.run(["whoami"], capture_output=True, text=True).stdout.strip()}')

## Install & Configure OpenSSH Server

In [ ]:
import subprocess, os

def run(cmd, check=True, capture=True):
    r = subprocess.run(cmd, shell=True, capture_output=capture, text=True)
    if check and r.returncode != 0:
        print(f'STDERR: {r.stderr.strip()}')
    return r

print('📦 Installing OpenSSH server...')
run('apt-get update -qq && apt-get install -y -qq openssh-server')
print('✅ OpenSSH installed')

# ── SSH daemon configuration ──────────────────────────────────────────────────
sshd_config = """
Port 22
AddressFamily any
ListenAddress 0.0.0.0
PermitRootLogin yes
PubkeyAuthentication yes
AuthorizedKeysFile /root/.ssh/authorized_keys
PasswordAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
X11Forwarding yes
PrintMotd no
AcceptEnv LANG LC_*
Subsystem sftp /usr/lib/openssh/sftp-server
# Allow longer keep-alive to survive Cloudflare idle timeouts
ClientAliveInterval 60
ClientAliveCountMax 10
"""

os.makedirs('/etc/ssh', exist_ok=True)
with open('/etc/ssh/sshd_config', 'w') as f:
    f.write(sshd_config)

# ── Host keys ─────────────────────────────────────────────────────────────────
run('ssh-keygen -A')   # generate all host keys if missing
print('✅ SSH host keys ready')

# ── Authorized key from Kaggle Secret ─────────────────────────────────────────
os.makedirs('/root/.ssh', exist_ok=True)
os.chmod('/root/.ssh', 0o700)

try:
    from kaggle_secrets import UserSecretsClient
    pub_key = UserSecretsClient().get_secret('SSH_PUBLIC_KEY').strip()
    with open('/root/.ssh/authorized_keys', 'w') as f:
        f.write(pub_key + '\n')
    os.chmod('/root/.ssh/authorized_keys', 0o600)
    print('✅ SSH public key loaded from Kaggle Secrets')
except Exception as e:
    print(f'⚠️  Could not load SSH_PUBLIC_KEY secret: {e}')
    print('   Add your public key manually below, or add it as a Kaggle Secret named SSH_PUBLIC_KEY')
    # ── FALLBACK: paste your public key here ──────────────────────────────────
    MANUAL_PUBLIC_KEY = ''  # <-- paste ssh-ed25519 AAAA... or ssh-rsa AAAA...
    # ─────────────────────────────────────────────────────────────────────────
    if MANUAL_PUBLIC_KEY:
        with open('/root/.ssh/authorized_keys', 'w') as f:
            f.write(MANUAL_PUBLIC_KEY + '\n')
        os.chmod('/root/.ssh/authorized_keys', 0o600)
        print('✅ Manual public key written')
    else:
        print('❌ No public key available — SSH login will fail!')

# ── Make conda/pip binaries reachable inside SSH sessions ─────────────────────
bashrc_additions = """
export PATH=/opt/conda/bin:/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH
export CUDA_HOME=/usr/local/cuda
"""
with open('/root/.bashrc', 'a') as f:
    f.write(bashrc_additions)

# Create symlinks so conda works immediately in SSH
for name in ['conda', 'python', 'pip', 'jupyter']:
    src = f'/opt/conda/bin/{name}'
    dst = f'/usr/local/bin/{name}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

# ── Start SSH daemon ──────────────────────────────────────────────────────────
run('service ssh start')
time.sleep(1)
status = run('service ssh status')
if 'active (running)' in status.stdout or 'is running' in status.stdout:
    print('✅ SSH daemon running on port 22')
else:
    print('⚠️  SSH daemon may not be running — check output:')
    print(status.stdout)

## Install cloudflared & Start Tunnel

In [ ]:
import subprocess, threading, time, re, os, json

CF_BIN = '/usr/local/bin/cloudflared'

if not os.path.exists(CF_BIN):
    print('📦 Downloading cloudflared...')
    subprocess.run(
        'curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/'
        'download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared '
        '&& chmod +x /usr/local/bin/cloudflared',
        shell=True, check=True
    )
    print('✅ cloudflared installed')
else:
    print('✅ cloudflared already present')

# ── Start a TCP quick-tunnel on port 22 ───────────────────────────────────────
# TryCloudflare gives a free *.trycloudflare.com hostname — no account needed.
# For production use, replace with: cloudflared tunnel run --token <TOKEN>

tunnel_url = None
tunnel_proc = None

def start_tunnel():
    global tunnel_url, tunnel_proc
    # --url tcp://localhost:22 exposes SSH via a TCP quick-tunnel
    cmd = [CF_BIN, 'tunnel', '--url', 'tcp://localhost:22',
           '--no-autoupdate', '--loglevel', 'info']
    tunnel_proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    hostname_re = re.compile(r'https://([a-z0-9\-]+\.trycloudflare\.com)')
    for line in tunnel_proc.stdout:
        m = hostname_re.search(line)
        if m:
            tunnel_url = m.group(1)
            break

t = threading.Thread(target=start_tunnel, daemon=True)
t.start()

print('⏳ Waiting for Cloudflare tunnel to initialise...')
for _ in range(30):
    if tunnel_url:
        break
    time.sleep(2)

if not tunnel_url:
    print('❌ Tunnel did not start in time. Check cloudflared output above.')
else:
    print(f'\n✅ Cloudflare tunnel active!')
    print(f'   Tunnel hostname : {tunnel_url}')
    print(f'   Proxying        : TCP → localhost:22 (SSH)\n')

    # ── Print GPU info one more time for the session log ──────────────────────
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader'], capture_output=True, text=True)
    gpu_label = gpu.stdout.strip() if gpu.returncode == 0 else 'CPU only'

    # ── Emit connection artefact ───────────────────────────────────────────────
    ssh_config_block = f"""# ── Add to ~/.ssh/config on your LOCAL machine ──────────────────
Host kaggle-gpu
    HostName {tunnel_url}
    User root
    Port 22
    IdentityFile ~/.ssh/id_ed25519        # path to your private key
    ProxyCommand cloudflared access tcp --hostname %h
    StrictHostKeyChecking no
    ServerAliveInterval 60
    ServerAliveCountMax 10
# ────────────────────────────────────────────────────────────────"""

    connection_info = {
        'tunnel_hostname': tunnel_url,
        'ssh_user': 'root',
        'port': 22,
        'proxy_command': f'cloudflared access tcp --hostname {tunnel_url}',
        'gpu': gpu_label,
        'ssh_config_alias': 'kaggle-gpu'
    }

    # Save connection info to disk for the pipeline connector to pick up
    with open('/kaggle/working/gpu_connection.json', 'w') as f:
        json.dump(connection_info, f, indent=2)

    print('━' * 66)
    print(ssh_config_block)
    print('━' * 66)
    print(f'\n🎮 GPU : {gpu_label}')
    print('\n📋 Connection JSON saved to /kaggle/working/gpu_connection.json')
    print('\n💡 On your local machine run:')
    print(f'   ssh kaggle-gpu')
    print('   (after adding the ssh_config block above to ~/.ssh/config)')

## Keep-alive (run this last, it blocks intentionally)

In [ ]:
import time, subprocess, datetime

print('🔒 Keep-alive loop started. This cell keeps the session alive.')
print('   Interrupt the kernel to stop.\n')

start = datetime.datetime.now()
interval = 60  # seconds between heartbeats

try:
    while True:
        elapsed = datetime.datetime.now() - start
        h, rem = divmod(int(elapsed.total_seconds()), 3600)
        m, s = divmod(rem, 60)
        gpu = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True
        )
        gpu_str = gpu.stdout.strip().replace('\n', ' | ') if gpu.returncode == 0 else 'N/A'
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'[{ts}] ⏱ Uptime {h:02d}h{m:02d}m | GPU util,mem_used,mem_total: {gpu_str}')
        time.sleep(interval)
except KeyboardInterrupt:
    print('\n⛔ Keep-alive stopped. Tunnel will close when the kernel stops.')